# BIOT 6900 · Module 9 — Toy Notebook
## Clinical NLP: watch the three gotchas fire
**Week 9 · Part III · conceptual warm-up (not graded)**

> **⚠️ SYNTHETIC DATA.** Every note in this notebook is invented for teaching. No real patient data appears anywhere here.

In lecture we said clinical text fights back three ways: **negation**, **abbreviation ambiguity**, and **temporality**. 
This notebook makes that concrete on six short notes so you can watch each one break a naive approach, then watch a simple fix handle it.

**Two promises.** Everything in the main sections runs on plain Python with **no installs**, so it works on any laptop right now. 
The real clinical-NLP tools (Med7, scispaCy, negspaCy) appear only at the very end as an optional preview, and come into full use in **Notebook 2**.

**How to run:** `Kernel → Restart & Run All`, then read the output under each cell. There is nothing to submit; the graded work is Notebook 3.

### Setup: the six synthetic notes
Read these first. Each one has a trap planted in it on purpose.

In [42]:
import re
import pandas as pd

# ----- SYNTHETIC notes (invented for teaching; NOT real patient data) -----
NOTES = [
    {"id": "N1", "text": "68M admitted with crushing substernal chest pain radiating to the left arm. Troponin elevated. Concern for acute MI; STEMI protocol activated."},
    {"id": "N2", "text": "72F with cough and fever. Patient denies chest pain. No evidence of pneumonia on CXR. Ruled out for MI."},
    {"id": "N3", "text": "55M here for routine follow-up. Mother has a history of breast cancer. No acute complaints."},
    {"id": "N4", "text": "34F with relapsing-remitting MS, new demyelinating lesion on brain MRI. Neuro exam at bedside."},
    {"id": "N5", "text": "Post-op day 1. Pain controlled with MS 2 mg IV q4h PRN. Tolerating diet."},
    {"id": "N6", "text": "81M with h/o MI and prior CVA. MS intact, alert and oriented x3. No chest pain today."},
]

print("Loaded", len(NOTES), "SYNTHETIC notes (not real patients).")
for n in NOTES:
    print(f"  {n['id']}: {n['text']}")


def sentences(text):
    return [s.strip() for s in re.split(r"[.;\n]", text) if s.strip()]

def contains(hay, needle):
    """Whole-word / whole-phrase, case-insensitive membership test."""
    return re.search(r"\b" + re.escape(needle) + r"\b", hay, flags=re.I) is not None

Loaded 6 SYNTHETIC notes (not real patients).
  N1: 68M admitted with crushing substernal chest pain radiating to the left arm. Troponin elevated. Concern for acute MI; STEMI protocol activated.
  N2: 72F with cough and fever. Patient denies chest pain. No evidence of pneumonia on CXR. Ruled out for MI.
  N3: 55M here for routine follow-up. Mother has a history of breast cancer. No acute complaints.
  N4: 34F with relapsing-remitting MS, new demyelinating lesion on brain MRI. Neuro exam at bedside.
  N5: Post-op day 1. Pain controlled with MS 2 mg IV q4h PRN. Tolerating diet.
  N6: 81M with h/o MI and prior CVA. MS intact, alert and oriented x3. No chest pain today.


## 1. The naive approach, and why it fails
The easy thing is to search the notes for a symptom. Let's look for **chest pain** and treat every note that contains the phrase as a patient who has it.

In [43]:
def naive_hits(term):
    return [n["id"] for n in NOTES if contains(n["text"], term)]

hits = naive_hits("chest pain")
print("Notes containing the phrase 'chest pain':", hits)
print("Naive conclusion: these", len(hits), "patients have chest pain.")

Notes containing the phrase 'chest pain': ['N1', 'N2', 'N6']
Naive conclusion: these 3 patients have chest pain.


Hold onto that number. Now read **N2** and **N6** again. One *denies* chest pain, the other has *no* chest pain today. 
Containing the phrase is not the same as the patient having the finding. That gap is the whole module.

## 2. Gotcha 1 — negation and assertion
The fix is to ask, for each mention, what the note actually *asserts*: is the finding **present**, **absent** (negated), **historical**, or about a **family member**? 
Below is a deliberately tiny, readable version of the classic **NegEx** idea: find a trigger word, and if it sits before the term in the same sentence, the term is negated. 
This is a teaching toy, not a production system (see the limitations note at the end).

In [44]:
NEG_TRIGGERS  = ["no evidence of", "no acute", "denies", "denied", "without",
                 "ruled out", "negative for", "r/o", "no "]
FAM_TRIGGERS  = ["mother", "father", "sister", "brother", "parent", "family history", "fh"]
HIST_TRIGGERS = ["history of", "h/o", "hx of", "prior", "previous", "status post", "s/p", "remote"]
CUR_TRIGGERS  = ["acute", "new", "today", "current", "now", "stat", "active"]

def assertion(term, text):
    """Return present | absent | historical | family for `term` in `text`."""
    for sent in sentences(text):
        if not contains(sent, term):
            continue
        low = sent.lower()
        before = low[:low.find(term.lower())]
        if any(t in low for t in FAM_TRIGGERS):      # belongs to someone else
            return "family"
        if any(t in before for t in NEG_TRIGGERS):   # negated before the term
            return "absent"
        if any(t in before for t in HIST_TRIGGERS):  # happened in the past
            return "historical"
        return "present"
    return None

rows = [{"note": n["id"], "naive_hit": True, "asserted_status": assertion("chest pain", n["text"])}
        for n in NOTES if contains(n["text"], "chest pain")]
df = pd.DataFrame(rows)
print(df.to_string(index=False))

present = [r["note"] for r in rows if r["asserted_status"] == "present"]
print(f"\nNaive said {len(rows)} patients with chest pain. Assertion-aware says {len(present)}: {present}")

note  naive_hit asserted_status
  N1       True         present
  N2       True          absent
  N6       True          absent

Naive said 3 patients with chest pain. Assertion-aware says 1: ['N1']


**There it is.** The naive count was 3; the assertion-aware count is 1. Two of the three were the model catching the word in a note that says the opposite. 
This single idea, negation detection, is the most important thing in the module.

### 2b. The same term, three different truths — negation *and* temporality
Now run the same check on **MI** (myocardial infarction). Watch one term come back three different ways across the notes.

In [45]:
rows = [{"note": n["id"], "mention": "MI", "status": assertion("MI", n["text"])}
        for n in NOTES if contains(n["text"], "MI")]
print(pd.DataFrame(rows).to_string(index=False))

note mention     status
  N1      MI    present
  N2      MI     absent
  N6      MI historical


N1 is an **acute MI** happening now. N2 was **ruled out** (absent). N6 is **h/o MI**, a heart attack in the past. 
Same two letters, opposite clinical meaning. That is Gotcha 3, temporality, falling out of the same machinery.

## 3. Gotcha 2 — abbreviation ambiguity
**MS** appears in three of our notes. It is not the same thing in any of them. A fixed lookup is wrong most of the time; only the surrounding words decide. 
Here is a toy context-based disambiguator: score each possible meaning by how many of its cue words show up in the note.

In [46]:
SENSES = {
    "multiple sclerosis": ["lesion", "demyelinat", "relapsing", "mri", "neuro", "plaque"],
    "mitral stenosis":    ["valve", "echo", "murmur", "regurg", "cardiac", "stenosis"],
    "morphine sulfate":   ["mg", "iv", "prn", "q4h", "q6h", "dose", "po"],
    "mental status":      ["intact", "alert", "oriented", "altered", "gcs", "exam", "confus"],
}

def disambiguate(abbr, text):
    low = text.lower()
    scores = {sense: sum(low.count(c) for c in cues) for sense, cues in SENSES.items()}
    best = max(scores, key=scores.get)
    return (best if scores[best] > 0 else "ambiguous"), {k: v for k, v in scores.items() if v}

rows = []
for n in NOTES:
    if contains(n["text"], "MS"):
        sense, cues = disambiguate("MS", n["text"])
        rows.append({"note": n["id"], "guess": sense, "cues_found": cues})
print(pd.DataFrame(rows).to_string(index=False))

note              guess                                    cues_found
  N4 multiple sclerosis {'multiple sclerosis': 5, 'mental status': 1}
  N5   morphine sulfate                       {'morphine sulfate': 5}
  N6      mental status                          {'mental status': 3}


N4 is multiple sclerosis, N5 is morphine sulfate, N6 is mental status. 
Notice N4 also picks up one weak cue for the wrong sense ("exam"): context scoring is better than a fixed lookup, but it is not perfect, which is exactly why the real models learn context instead of counting keywords.

## 4. From notes to a feature table
Put it together. For a handful of concepts, record the asserted status per note. This little grid is the shape of the feature matrix you will build for real in Notebook 2, where it joins the structured data (labs, vitals) and feeds an outcome model.

In [47]:
concepts = ["chest pain", "MI", "pneumonia", "breast cancer"]
grid = []
for n in NOTES:
    row = {"note": n["id"]}
    for c in concepts:
        row[c] = assertion(c, n["text"]) if contains(n["text"], c) else "-"
    grid.append(row)
print(pd.DataFrame(grid).to_string(index=False))

note chest pain         MI pneumonia breast cancer
  N1    present    present         -             -
  N2     absent     absent    absent             -
  N3          -          -         -        family
  N4          -          -         -             -
  N5          -          -         -             -
  N6     absent historical         -             -


Read the `breast cancer` column: N3 is **family**, not the patient. A naive feature would have marked this patient positive. The asserted status is the feature you actually want.

## 5. Optional preview — the real tools
Everything above was from scratch, to show the machinery. In practice you use purpose-built tools, which is what **Notebook 2** sets up. 
The two cells below are a preview. If the libraries are not installed yet they will skip cleanly, and the notebook still finishes. **You do not need to install anything for this toy.**

In [48]:
# Preview: negspaCy runs the real NegEx on detected entities.
try:
    import spacy
    from negspacy.negation import Negex
    nlp = spacy.load("en_core_web_sm")
    nlp.add_pipe("negex")
    doc = nlp("Patient denies chest pain but reports fever.")
    for e in doc.ents:
        print(f"{e.text:20s} negated={e._.negex}")
except Exception as ex:
    print("[optional] real negation tools not installed here (" + type(ex).__name__ + ").")
    print("Expected: the from-scratch cells above already ran. Notebook 2 installs and uses negspaCy.")

In [49]:
# Preview: Med7 pulls medication details (drug, dose, route, frequency...) from text.
try:
    import spacy
    nlp = spacy.load("en_core_med7_lg")
    doc = nlp("Pain controlled with morphine sulfate 2 mg IV q4h PRN.")
    print([(e.text, e.label_) for e in doc.ents])
except Exception as ex:
    print("[optional] Med7 not installed here (" + type(ex).__name__ + ").")
    print("Notebook 2 installs Med7 and uses it to extract drug / dose / route / frequency.")

[optional] Med7 not installed here (OSError).
Notebook 2 installs Med7 and uses it to extract drug / dose / route / frequency.


## Recap
- **Negation** turned 3 naive hits into 1 real one. Naming a finding is often how clinicians record that it is *absent*.
- **Abbreviation**: `MS` meant three different things; only context decided.
- **Temporality**: `MI` came back present, absent, and historical from the same code.
- The asserted-status grid is the beginning of a feature table.

**Limitations of this toy** (good to say out loud): sentence-scoped triggers miss double negatives and pseudo-negation ("no *change*"), the cue lists are hand-made, and there is no real entity recognition. The real tools in Notebook 2 handle these far better, and your job is to check where they still get it wrong.

**The one idea to carry:** the table says what was *measured*; the note says what was *thought*. Next, Notebook 2 runs the real pipeline on real structured data and builds an outcome model.